In [16]:
# Cargar json de resultados de algo_comparison/20220103
# - results_dc.json
# - wct_results.json
# - cc_static_results.json

from pathlib import Path
import json
from loguru import logger
import numpy as np
from collections import defaultdict
import copy

results_path = Path("/workspaces/SOLhycool/optimization/notebooks/algo_comparison/20220103")

results_dict = {
    system_id: json.load(open(results_path / f"{system_id}_results.json", "r")) 
    for system_id in ["dc", "wct", "cc_static"]
}

print( results_dict["dc"][0].keys() )
results_dict["dc"][0]["ihs_50_800"].keys()


dict_keys(['ihs_50_800', 'ihs_100_800', 'ihs_400_800', 'sea_cstr_50_80', 'sea_cstr_100_80', 'sea_cstr_400_80', 'de_cstr_50_1', 'de_cstr_100_0', 'de_cstr_400_0'])


dict_keys(['algo_id', 'params', 'avg_fitness', 'var_fitness', 'avg_n_obj_fun_evals', 'fitness_history'])

In [10]:
results_dict["dc"][0]["ihs_50_800"]["avg_n_obj_fun_evals"]


850.0

In [11]:
results_dict["dc"][0]["ihs_50_800"]["params"]


{'pop_size': 50, 'max_iter': 800, 'cstr_sa': False}

In [12]:
results_dict["dc"][0]["ihs_50_800"]["avg_fitness"]


0.9727111213951786

In [13]:
results_dict["dc"][0]["ihs_50_800"]["var_fitness"]


0.0

In [22]:
results_dict["dc"][0]["ihs_50_800"]["fitness_history"]


[1.1106586119822885,
 1.1106586119822885,
 1.1096765228978323,
 1.108694433813376,
 1.1077123447289199,
 1.1067302556444636,
 1.1057481665600073,
 1.104766077475551,
 1.103783988391095,
 1.1028018993066386,
 1.1018198102221823,
 1.100837721137726,
 1.09985563205327,
 1.0988735429688137,
 1.0978914538843574,
 1.096909364799901,
 1.095927275715445,
 1.0949451866309887,
 1.0939630975465324,
 1.0929810084620761,
 1.09199891937762,
 1.0910168302931638,
 1.0900347412087075,
 1.0890526521242512,
 1.088070563039795,
 1.0870884739553388,
 1.0861063848708825,
 1.0851242957864262,
 1.08414220670197,
 1.0831601176175139,
 1.0821780285330576,
 1.0811959394486013,
 1.080213850364145,
 1.079231761279689,
 1.0782496721952326,
 1.0772675831107763,
 1.07628549402632,
 1.075303404941864,
 1.0743213158574076,
 1.0733392267729513,
 1.072357137688495,
 1.071375048604039,
 1.0703929595195827,
 1.0694108704351264,
 1.06842878135067,
 1.067446692266214,
 1.0664646031817577,
 1.0654825140973014,
 1.064500425012

In [28]:
data = copy.deepcopy(results_dict["dc"])

def process_evaluations(data):
    aggregated = defaultdict(lambda: {
        'algo_id': None,
        'params': None,
        'avg_n_obj_fun_evals_list': [],
        'avg_fitness_list': [],
        'fitness_history_list': [],
    })

    for evaluation in data:
        for alt_key, alt_data in evaluation.items():
            entry = aggregated[alt_key]

            # Set common fields (same across all, so we just use the first one encountered)
            entry['algo_id'] = alt_data['algo_id']
            entry['params'] = alt_data['params']

            # Accumulate lists
            entry['avg_n_obj_fun_evals_list'].append(alt_data['avg_n_obj_fun_evals'])
            entry['avg_fitness_list'].append(alt_data['avg_fitness'])
            entry['fitness_history_list'].append(alt_data['fitness_history'])

    # Now compute the final aggregated dictionary
    final_result = {}

    for alt_key, entry in aggregated.items():
        histories = entry['fitness_history_list']
        max_len = max(len(h) for h in histories)
        
        # Pad with np.nan to the right to make all histories the same length
        padded_histories = [
            np.pad(h, (0, max_len - len(h)), constant_values=np.nan) for h in histories
        ]
        
        fitness_history_array = np.array(padded_histories)
        
        fitness_avg = np.nanmean(entry['avg_fitness_list'])
        fitness_history_avg = np.nanmean(fitness_history_array, axis=0)
        fitness_var = np.nanvar(entry['avg_fitness_list'], ddof=0)
        fitness_history_var = np.nanvar(fitness_history_array, axis=0, ddof=0)

        final_result[alt_key] = {
            'algo_id': entry['algo_id'],
            'params': entry['params'],
            'avg_n_obj_fun_evals': np.nanmean(entry['avg_n_obj_fun_evals_list']),
            'avg_fitness': fitness_avg,
            'var_fitness': fitness_var,
            'fitness_history': fitness_history_avg.tolist(),
            'var_fitness_history': fitness_history_var.tolist()
        }

    return final_result

table_data = {system_id: process_evaluations(data) for system_id, data in results_dict.items()}


In [106]:
table_header = r"""
\begin{table}[]
\begin{tabular}{llllclclccccccccc}
\hline
\multirow{2}{*}{\textbf{System}} &  & \multicolumn{1}{c}{\multirow{2}{*}{\textbf{Algorithm}}} &  & \multicolumn{5}{c}{\textbf{Parameters}}                                                                                             &  & \multicolumn{7}{c}{\textbf{Average fitness per obj. fun. evaluations}}                                                                 \\ \cline{5-9} \cline{11-17} 
                                 &  & \multicolumn{1}{c}{}                                    &  & \textbf{\begin{tabular}[c]{@{}c@{}}pop\\ size\end{tabular}} &  & \textbf{gen}     &  & \textbf{\begin{tabular}[c]{@{}c@{}}wrapper\\ algo\\ iters\end{tabular}} &  & \multicolumn{1}{c}{\textbf{0}} & \textbf{} & \multicolumn{1}{c}{\textbf{50}} & \textbf{} & \multicolumn{1}{c}{\textbf{150}} & \textbf{} & \multicolumn{1}{c}{\textbf{800}} \\ \cline{1-1} \cline{3-3} \cline{5-5} \cline{7-7} \cline{9-9} \cline{11-11} \cline{13-13} \cline{15-15} \cline{17-17} 
"""

table_footer = r"""
\end{tabular}
\end{table}
"""

body_template = r"""
\multirow{9}{*}{system-id}       &  & \multirow{3}{*}{algo-id-1}                              &  & pop-size-1-1                                       &  & gen-1-1 &  & w-iters-1-1                                                    &  & val-1-1-1             &           & val-1-1-2              &           & val-1-1-3               &           & val-1-1-4               \\
                                 &  &                                                         &  & pop-size-1-2                                       &  & gen-1-2 &  & w-iters-1-2                                                    &  & val-1-2-1             &           & val-1-2-2              &           & val-1-2-3               &           & val-1-2-4               \\
                                 &  &                                                         &  & pop-size-1-3                                       &  & gen-1-3 &  & w-iters-1-3                                                    &  & val-1-3-1             &           & val-1-3-2              &           & val-1-3-3               &           & val-1-3-4               \\ \cline{3-3} \cline{5-9} \cline{11-17} 
                                 &  & \multirow{3}{*}{algo-id-2}                              &  & pop-size-2-1                                       &  & gen-2-1 &  & w-iters-2-1                                                    &  & val-2-1-1             &           & val-2-1-2              &           & val-2-1-3               &           & val-2-1-4               \\ 
                                 &  &                                                         &  & pop-size-2-2                                       &  & gen-2-2 &  & w-iters-2-2                                                    &  & val-2-2-1             &           & val-2-2-2              &           & val-2-2-3               &           & val-2-2-4               \\
                                 &  &                                                         &  & pop-size-2-3                                       &  & gen-2-3 &  & w-iters-2-3                                                    &  & val-2-3-1             &           & val-2-3-2              &           & val-2-3-3               &           & val-2-3-4               \\ \cline{3-3} \cline{5-9} \cline{11-17} 
                                 &  & \multirow{3}{*}{algo-id-3}                              &  & pop-size-3-1                                       &  & gen-3-1 &  & w-iters-3-1                                                    &  & val-3-1-1             &           & val-3-1-2              &           & val-3-1-3               &           & val-3-1-4               \\ 
                                 &  &                                                         &  & pop-size-3-2                                       &  & gen-3-2 &  & w-iters-3-2                                                    &  & val-3-2-1             &           & val-3-2-2              &           & val-3-2-3               &           & val-3-2-4               \\
                                 &  &                                                         &  & pop-size-3-3                                       &  & gen-3-3 &  & w-iters-3-3                                                    &  & val-3-3-1             &           & val-3-3-2              &           & val-3-3-3               &           & val-3-3-4               \\ \hline
"""


In [107]:
algo_ids = list(set([cs_vals["algo_id"] for cs_vals in list(table_data.values())[0].values()]))
obj_fun_vals = [0, 50, 150, 750]
system_labels = {
    "dc": "DC",
    "wct": "WCT",
    "cc_static": "CC"
}

body_content = []

for system_id, data in table_data.items():
    body_cnt = copy.deepcopy(body_template).replace("system-id", "\\textbf{"+system_labels[system_id]+"}")
    
    for algo_idx, algo_id in enumerate(algo_ids):
        body_cnt = body_cnt.replace(f"algo-id-{algo_idx+1}", algo_id)
        
        cs_idx = 1
        for cs_data in data.values():
            # print(cs_data["algo_id"], algo_id)
            if cs_data["algo_id"] != algo_id:
                continue
        
            # print(f"Processing {system_id} - {algo_id} - {algo_idx+1} - {cs_idx}")
        
            # Algo parameters
            params = cs_data['params']
            pop_size = params.get('pop_size', 'N/A')
            
            # Chapuza
            if params.get('cstr_sa', None):
                wrapper_algo_iters = 10 
            else:
                wrapper_algo_iters = 'N/A'
            
            gen = params["max_iter"]
            # if wrapper_algo_iters != 'N/A':
            #     gen = gen // wrapper_algo_iters
            # if algo_id not in ["ihs", "sea"]:
            #     gen = gen // pop_size

            body_cnt = body_cnt.replace(f"pop-size-{algo_idx+1}-{cs_idx}", str(pop_size))
            body_cnt = body_cnt.replace(f"gen-{algo_idx+1}-{cs_idx}", str(gen))
            body_cnt = body_cnt.replace(f"w-iters-{algo_idx+1}-{cs_idx}", str(wrapper_algo_iters))

            # Fitness values
            for fit_idx, obj_fun_val in enumerate(obj_fun_vals):
                fitness_value = cs_data["fitness_history"][obj_fun_val] if len(cs_data["fitness_history"]) > obj_fun_val else cs_data["fitness_history"][-1]
                fitness_var = cs_data["var_fitness_history"][obj_fun_val] if len(cs_data["var_fitness_history"]) > obj_fun_val else cs_data["fitness_history"][-1]
                body_cnt = body_cnt.replace(
                    f"val-{algo_idx+1}-{cs_idx}-{fit_idx+1}", 
                    f"${fitness_value:.2f}\\pm{fitness_var:.2f}$"
                )
                
            # for i, fitness in enumerate(algo_data['avg_fitness_history']):
            #     body_cnt = body_cnt.replace(f"val-1-1-{i+1}", f"{fitness:.2f}")
            
            cs_idx += 1

    body_content.append(body_cnt)

print(body_content[0])



\multirow{9}{*}{\textbf{DC}}       &  & \multirow{3}{*}{ihs}                              &  & 50                                       &  & 800 &  & N/A                                                    &  & $1.28\pm0.82$             &           & $1.05\pm0.29$              &           & $0.80\pm0.10$               &           & $0.77\pm0.09$               \\
                                 &  &                                                         &  & 100                                       &  & 800 &  & N/A                                                    &  & $0.92\pm0.18$             &           & $0.87\pm0.14$              &           & $0.81\pm0.11$               &           & $0.77\pm0.10$               \\
                                 &  &                                                         &  & 400                                       &  & 800 &  & N/A                                                    &  & $0.81\pm0.11$             &           & $0.80\pm0.1

In [108]:
table_str = f"""{table_header}
{body_content[0]}
{body_content[1]}
{body_content[2]}
{table_footer}"""

print(table_str)



\begin{table}[]
\begin{tabular}{llllclclccccccccc}
\hline
\multirow{2}{*}{\textbf{System}} &  & \multicolumn{1}{c}{\multirow{2}{*}{\textbf{Algorithm}}} &  & \multicolumn{5}{c}{\textbf{Parameters}}                                                                                             &  & \multicolumn{7}{c}{\textbf{Average fitness per obj. fun. evaluations}}                                                                 \\ \cline{5-9} \cline{11-17} 
                                 &  & \multicolumn{1}{c}{}                                    &  & \textbf{\begin{tabular}[c]{@{}c@{}}pop\\ size\end{tabular}} &  & \textbf{gen}     &  & \textbf{\begin{tabular}[c]{@{}c@{}}wrapper\\ algo\\ iters\end{tabular}} &  & \multicolumn{1}{c}{\textbf{0}} & \textbf{} & \multicolumn{1}{c}{\textbf{50}} & \textbf{} & \multicolumn{1}{c}{\textbf{150}} & \textbf{} & \multicolumn{1}{c}{\textbf{800}} \\ \cline{1-1} \cline{3-3} \cline{5-5} \cline{7-7} \cline{9-9} \cline{11-11} \cline{13-13} \cline{15-15} \